In [1]:
import polars as pl
from aeon.classification.convolution_based import MiniRocketClassifier, RocketClassifier
from aeon.classification.feature_based import (
    Catch22Classifier,
)
from aeon.datasets.tsc_datasets import univariate_equal_length
from sklearn.metrics import accuracy_score

from autotsc.models import AutoTSCModel
from autotsc.utils import load_dataset

KeyboardInterrupt: 

In [ ]:
stats = []
selected_datasets = univariate_equal_length  # [::3]

In [ ]:
import ray

ray.shutdown()
ray.init(num_cpus=24)

In [ ]:
def get_model(name):
    if name == "Catch22Classifier":
        return Catch22Classifier(n_jobs=-1)
    elif name == "RocketClassifier":
        return RocketClassifier(n_jobs=-1)
    elif name == "MiniRocketClassifier":
        return MiniRocketClassifier(n_jobs=-1)
    elif name == "AutoTSCModel":
        return AutoTSCModel(
            n_jobs=-1,
            model_types="catch22",
            verbose=True,
            use_stacking=False,
            use_only_best_model=False,
        )
    else:
        raise ValueError(f"Unknown model name: {name}")


for dataset in selected_datasets:
    X_train, y_train, X_test, y_test = load_dataset(dataset)

    for model_name in [
        "Catch22Classifier",
        "RocketClassifier",
        "MiniRocketClassifier",
        "AutoTSCModel",
    ]:
        try:
            model = get_model(model_name)
            model.fit(X_train, y_train)
            pred = model.predict(X_test)
            test_acc = accuracy_score(y_test, pred)
            stats.append(
                {
                    "dataset": dataset,
                    "model": model_name,
                    "test_accuracy": test_acc,
                }
            )
        except Exception as e:
            print(f"Error with dataset {dataset}: {e}")

In [ ]:
stats = pl.DataFrame(stats)
stats

In [ ]:
# transform model column values into new columns
stats_pivot = stats.pivot(values="test_accuracy", index="dataset", columns="model").drop_nulls()
stats_pivot

In [ ]:
models = stats["model"].unique().to_list()
P = stats_pivot.select(models).to_numpy()

In [ ]:
from aeon.visualisation import plot_critical_difference

plot_critical_difference(P, models)